# An agent whose model and tools run on your Mac

A LangGraph agent answers questions about a database. Its brain is Qwen3-4B served on this Mac's GPU by a Flyte app; its tools are Flyte tasks, so every query becomes a durable child action. No API key.

| Step | What it shows |
|---|---|
| 0 | Serve the brain. |
| 1 | The tools, with no model anywhere. |
| 2 | One question, with the tool-call trace. |
| 3 | Sixteen questions with known answers, fanned out and scored. |

---

## Setup

This notebook runs **on your Mac**, next to an Apple Silicon devbox. It is not a Colab notebook: Colab has no Metal GPU and no devbox. (The code still runs there with `flyte run --local` and `ACCELERATOR=cpu`, just slowly.)

Before the first cell:

```bash
flyte start devbox --apple-silicon
cd tutorials/apple-silicon-agent
flyte create config --devbox
uv venv .venv --python 3.12 && source .venv/bin/activate && uv pip install -r requirements.txt pygments jupyter
```

In [ ]:
import os, platform
assert platform.machine() == 'arm64' and platform.system() == 'Darwin', 'This notebook expects an Apple Silicon Mac'
from utils.file_viewer import view_file
from utils.report_viewer import run_link, show_latest_local
!flyte get devbox --no-probes | head -5

### Running the steps

| | |
|---|---|
| `flyte run ...` | On the devbox. GPU tasks run natively on this Mac; every run gets a page in the UI at http://localhost:30080 with logs and reports. |
| `flyte run --local ...` | In this process, no devbox. On a Mac the GPU tasks still use `mps`. Reports land under `/tmp/flyte/metadata`; `show_latest_local()` renders the newest one. |

The cells below use the devbox. Each prints a run URL: open it for the live report.

### 0. Serve the brain

Qwen3-4B is about 8 GB in bf16. On a 16 GB Mac use `MODEL_NAME=Qwen/Qwen3-1.7B` for this cell and the `flyte run` cells below.

In [ ]:
!python serve_model.py

### 1. The tools, with no model anywhere

Each is an `@env.task`. `run_sql` is read-only and capped at 50 rows.

In [ ]:
!flyte run tools.py schema

In [ ]:
!flyte run tools.py run_sql --query "SELECT Country, COUNT(*) n FROM Customer GROUP BY Country ORDER BY n DESC LIMIT 3"

In [ ]:
view_file("tools.py", title="the tools", show_path=True)

### 2. One question

Open the run: the agent task has a child per tool call, with the query as input and the rows as output. The report shows the same trace.

In [ ]:
!flyte run agent.py ask --question "Which artist has the most albums?"

In [ ]:
view_file("agent.py", title="the loop, the tasks, the scorer", show_path=True)

### 3. Sixteen questions at once

In [ ]:
!flyte run agent.py evaluate

### Run the agent here, against the model on the GPU

`--local` runs the loop in this process. The model stays where it is; `LLM_BASE_URL` tells the agent where.

In [ ]:
%env LLM_BASE_URL=http://apple-silicon-agent-llm-flytesnacks-development.localhost:30081
!flyte run --local agent.py ask --question "How many customers are from Brazil?"
show_latest_local()

### Swap the brain, or move it

```bash
AGENT_MODEL=anthropic:claude-haiku-4-5 flyte run agent.py evaluate   # after: flyte create secret ANTHROPIC_API_KEY --value ...
ACCELERATOR=L40s python serve_model.py && ACCELERATOR=L40s flyte run agent.py evaluate   # on a cluster
```